# Speak with the PDF — RAG Mini App
**Module 3 Assignment | Google Colab | OpenRouter**

This notebook implements a complete PDF question-answering pipeline:
1. Upload a PDF and extract its text.
2. Split the text into deterministic overlapping chunks.
3. Generate embeddings through OpenRouter and build a local cosine-similarity index (Pinecone is optional).
4. Compare a vanilla LLM answer with a retrieval-augmented (RAG) answer for three questions.

**Important:** Create your own copy before editing. Never paste an API key into a code cell or share it in screenshots. The notebook asks for the key privately using `getpass`.

**Models:** The defaults below use OpenRouter model IDs. If a model is unavailable to your account, change the model ID in the configuration cell to an embedding/chat model listed in your OpenRouter account.

## 1. Install dependencies

In [2]:
%pip -q install openai pypdf "numpy<2.5" "pandas==2.2.3"

## 2. Imports and secure OpenRouter setup

In [3]:
import os
import re
import math
import hashlib
from typing import Any, Dict, List
from getpass import getpass

import numpy as np
import pandas as pd
from pypdf import PdfReader
from openai import OpenAI

# Enter the OpenRouter key privately when prompted; it is not displayed.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ").strip()
if not OPENROUTER_API_KEY:
    raise ValueError("An OpenRouter API key is required.")

# OpenAI-compatible SDK pointed at OpenRouter.
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Speak with PDF - Module 3 Assignment"
    }
)

# Change these IDs only if the defaults are unavailable in your OpenRouter account.
EMBEDDING_MODEL = "openai/text-embedding-3-small"
CHAT_MODEL = "openai/gpt-4o-mini"

print("OpenRouter client configured.")
print("Embedding model:", EMBEDDING_MODEL)
print("Chat model:", CHAT_MODEL)

Enter your OpenRouter API key: ··········
OpenRouter client configured.
Embedding model: openai/text-embedding-3-small
Chat model: openai/gpt-4o-mini


## 3. PDF parsing and chunking

`parse_pdf_to_chunks()` extracts text from every readable page, normalizes whitespace, and creates deterministic overlapping character chunks. It handles empty pages and rejects invalid chunk settings.

In [4]:
def parse_pdf_to_chunks(pdf_path: str, chunk_size: int = 800, overlap: int = 120) -> List[str]:
    """Extract PDF text and return ordered, non-empty overlapping chunks.

    Chunk boundaries are deterministic. Each chunk is at most chunk_size
    characters; overlap characters are repeated at the start of the next chunk.
    """
    if not isinstance(chunk_size, int) or chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer.")
    if not isinstance(overlap, int) or overlap < 0:
        raise ValueError("overlap must be a non-negative integer.")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")
    if not os.path.isfile(pdf_path):
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    reader = PdfReader(pdf_path)
    page_texts = []
    for page in reader.pages:
        try:
            text = page.extract_text() or ""
        except Exception:
            text = ""
        if text.strip():
            page_texts.append(text)

    raw_text = "\n".join(page_texts)
    # Normalize whitespace without destroying paragraph boundaries entirely.
    raw_text = raw_text.replace("\x00", " ")
    raw_text = re.sub(r"[ \t]+", " ", raw_text)
    raw_text = re.sub(r"\s*\n\s*", " ", raw_text)
    raw_text = re.sub(r"\s+", " ", raw_text).strip()

    if not raw_text:
        raise ValueError("No extractable text was found. The PDF may be scanned; run OCR first.")

    chunks = []
    start = 0
    text_length = len(raw_text)
    while start < text_length:
        end = min(start + chunk_size, text_length)
        # Prefer a nearby word boundary, while keeping chunks within chunk_size.
        if end < text_length:
            boundary = raw_text.rfind(" ", start + int(chunk_size * 0.7), end)
            if boundary > start:
                end = boundary
        chunk = raw_text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_length:
            break
        start = max(0, end - overlap)

    return chunks

print("PDF chunking function is ready.")

PDF chunking function is ready.


## 4. Embeddings and vector indexing

The default backend is a local NumPy vector index, so no Pinecone account is required. Set `use_pinecone=True` only if Pinecone credentials and package are configured. Embeddings are generated in batches through OpenRouter.

In [5]:
def _embed_texts(texts: List[str], model: str, batch_size: int = 64) -> np.ndarray:
    """Create embeddings through the OpenRouter embeddings endpoint."""
    if not texts:
        raise ValueError("Cannot embed an empty list.")
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        # Sort by response index to preserve input order.
        batch_data = sorted(response.data, key=lambda item: item.index)
        vectors.extend([item.embedding for item in batch_data])
    matrix = np.asarray(vectors, dtype=np.float32)
    if matrix.ndim != 2 or matrix.shape[0] != len(texts):
        raise RuntimeError("Embedding response had an unexpected shape.")
    return matrix


def _normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)


def embed_and_index(
    chunks: List[str],
    use_pinecone: bool = True,
    model: str = "openai/text-embedding-3-small"
) -> dict:
    """Embed chunks and create a Pinecone or local vector index.

    Returns the required fields: chunks, index_backend, index_ref,
    embed_model, and dim. If Pinecone is requested but not configured,
    the function safely uses the local in-memory index.
    """
    if not chunks:
        raise ValueError("chunks cannot be empty.")
    clean_chunks = [str(chunk).strip() for chunk in chunks if str(chunk).strip()]
    if not clean_chunks:
        raise ValueError("No non-empty chunks were supplied.")

    embeddings = _embed_texts(clean_chunks, model=model)
    normalized = _normalize_rows(embeddings)
    dim = int(embeddings.shape[1])

    # Optional Pinecone backend. Local is the reliable default for this assignment.
    pinecone_key = os.getenv("PINECONE_API_KEY", "").strip()
    if use_pinecone and pinecone_key:
        try:
            from pinecone import Pinecone, ServerlessSpec
            pc = Pinecone(api_key=pinecone_key)
            index_name = os.getenv("PINECONE_INDEX_NAME", "speak-with-pdf").lower().replace("_", "-")
            existing = [item["name"] if isinstance(item, dict) else item.name for item in pc.list_indexes()]
            if index_name not in existing:
                pc.create_index(
                    name=index_name,
                    dimension=dim,
                    metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region=os.getenv("PINECONE_REGION", "us-east-1"))
                )
            pine_index = pc.Index(index_name)
            records = []
            for i, vector in enumerate(embeddings):
                stable_id = hashlib.sha256(f"{i}:{clean_chunks[i]}".encode("utf-8")).hexdigest()[:32]
                records.append({
                    "id": stable_id,
                    "values": vector.astype(float).tolist(),
                    "metadata": {"chunk": clean_chunks[i], "order": i}
                })
            for start in range(0, len(records), 100):
                pine_index.upsert(vectors=records[start:start + 100])
            return {
                "chunks": clean_chunks,
                "index_backend": "pinecone",
                "index_ref": {"index": pine_index, "name": index_name},
                "embed_model": model,
                "dim": dim,
                "embeddings": embeddings,
                "normalized_embeddings": normalized
            }
        except Exception as exc:
            print("Pinecone setup failed; continuing with local index.")
            print("Reason:", str(exc)[:300])

    return {
        "chunks": clean_chunks,
        "index_backend": "local",
        "index_ref": {"matrix": normalized, "ids": list(range(len(clean_chunks)))},
        "embed_model": model,
        "dim": dim,
        "embeddings": embeddings,
        "normalized_embeddings": normalized
    }

print("Embedding and indexing function is ready.")

Embedding and indexing function is ready.


## 5. Retrieval and RAG answer generation

The question is embedded with the same embedding model used for the PDF. The top-k chunks are selected using cosine similarity. The LLM is instructed to answer only from retrieved context.

In [6]:
def retrieve_chunks(question: str, index_obj: dict, top_k: int = 3):
    """Return (chunk, cosine_score) pairs in descending similarity order."""
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")
    if not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    q_vector = _embed_texts([question], model=index_obj["embed_model"])[0]
    q_vector = q_vector / max(float(np.linalg.norm(q_vector)), 1e-12)

    if index_obj["index_backend"] == "local":
        matrix = index_obj["index_ref"]["matrix"]
        scores = matrix @ q_vector
        # Stable ordering: similarity descending, then original chunk index ascending.
        order = np.lexsort((np.arange(len(scores)), -scores))[:min(top_k, len(scores))]
        return [(index_obj["chunks"][int(i)], float(scores[i])) for i in order]

    if index_obj["index_backend"] == "pinecone":
        pine_index = index_obj["index_ref"]["index"]
        result = pine_index.query(
            vector=q_vector.astype(float).tolist(),
            top_k=min(top_k, len(index_obj["chunks"])),
            include_metadata=True
        )
        matches = []
        for match in result.matches:
            metadata = match.metadata or {}
            chunk = metadata.get("chunk", "")
            if chunk:
                matches.append((chunk, float(match.score)))
        return matches

    raise ValueError("Unknown index backend.")


def answer_query(
    question: str,
    index_obj: dict,
    top_k: int = 3,
    gpt_model: str = "openai/gpt-4o-mini"
) -> str:
    """Retrieve relevant PDF chunks and generate a context-grounded answer."""
    retrieved = retrieve_chunks(question, index_obj, top_k=top_k)
    context = "\n\n--- Retrieved chunk ---\n\n".join(chunk for chunk, score in retrieved)
    if not context:
        return "I don't know based on the document."

    response = client.chat.completions.create(
        model=gpt_model,
        temperature=0.1,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use ONLY the provided context to answer. "
                    "If the answer is not supported by the context, respond exactly: "
                    "'I don't know based on the document.' Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": f"Question: {question}\n\nContext:\n{context}"
            }
        ]
    )
    answer = response.choices[0].message.content
    return (answer or "").strip()

print("Retrieval and answer functions are ready.")

Retrieval and answer functions are ready.


## 6. Upload and process a PDF

Run this cell and select a PDF from your computer. For scanned/image-only PDFs, OCR is required before text extraction.

In [8]:
from google.colab import files

uploaded = files.upload()
pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
if not pdf_files:
    raise ValueError("Please upload at least one .pdf file.")

PDF_PATH = pdf_files[0]
chunks = parse_pdf_to_chunks(PDF_PATH, chunk_size=800, overlap=120)

print("PDF:", PDF_PATH)
print("Number of chunks:", len(chunks))
print("\nFirst chunk preview:\n", chunks[0][:700])

Saving HCLTech_NVIDIA_Complete_Briefing.pdf to HCLTech_NVIDIA_Complete_Briefing.pdf
PDF: HCLTech_NVIDIA_Complete_Briefing.pdf
Number of chunks: 29

First chunk preview:
 HCLTech × NVIDIA Complete Company, Partnership & Strategy Briefing A practical briefing for a university vendor-selection role-play Includes Company overview • Partnership facts • Technology ecosystem • Timeline • Solutions • $200M proposal framework • Risks • Panel Q&A • Executive pitch Prepared: 13 October 2026 Important distinction: publicly documented partnership activity is factual background. The proposed $200 million, three-year program described in this document is a role-play proposal, not an assertion that HCLTech and NVIDIA have announced or signed that specific contract. HCLTech × NVIDIA | Briefing Page 2 Executive summary HCLTech is a global IT services, consulting and engineeri


## 7. Build the vector index

In [9]:
# Local indexing works without Pinecone credentials.
# Change to True only after configuring PINECONE_API_KEY and installing pinecone.
USE_PINECONE = False

index_obj = embed_and_index(
    chunks,
    use_pinecone=USE_PINECONE,
    model=EMBEDDING_MODEL
)

print("Index backend:", index_obj["index_backend"])
print("Embedding model:", index_obj["embed_model"])
print("Vector dimensions:", index_obj["dim"])
print("Indexed chunks:", len(index_obj["chunks"]))

Index backend: local
Embedding model: openai/text-embedding-3-small
Vector dimensions: 1536
Indexed chunks: 29


## 8. Ask one question (interactive mini app)

Enter any question about the uploaded PDF. The answer will be grounded in retrieved chunks.

In [10]:
question = input("Ask a question about your PDF: ").strip()
if question:
    print("\nRAG ANSWER\n")
    print(answer_query(question, index_obj, top_k=3, gpt_model=CHAT_MODEL))
else:
    print("No question entered.")

Ask a question about your PDF: what is the deal

RAG ANSWER

The deal refers to a proposed $200 million, three-year program between HCLTech and NVIDIA, organized into phases of Build, Scale, and Optimize. However, it is important to note that this proposal is part of a role-play scenario and has not been officially announced or signed as a contract. The focus of the partnership is on integrating NVIDIA's computing and AI platforms into enterprise environments to create measurable business value.


## 9. Comparative playtests — three questions

Edit the three questions below so they are specific to your uploaded PDF. This cell records both vanilla LLM and RAG answers for each question. The vanilla answer receives no PDF context; the RAG answer receives retrieved chunks.

In [11]:
def vanilla_answer(question: str, gpt_model: str = CHAT_MODEL) -> str:
    """Answer without PDF retrieval (baseline comparison)."""
    response = client.chat.completions.create(
        model=gpt_model,
        temperature=0.1,
        messages=[
            {"role": "system", "content": "Answer the user's question using your general knowledge. If uncertain, say so."},
            {"role": "user", "content": question}
        ]
    )
    return (response.choices[0].message.content or "").strip()


# Replace these examples with three domain-specific questions about your PDF.
TEST_QUESTIONS = [
    "What is the main purpose or central argument of this document?",
    "What are the most important methods, steps, or components described?",
    "What key findings, conclusions, or recommendations does the document present?"
]

comparison_rows = []
for number, q in enumerate(TEST_QUESTIONS, start=1):
    print("\n" + "=" * 90)
    print(f"QUESTION {number}: {q}")
    print("\n--- VANILLA LLM (no PDF retrieval) ---")
    vanilla = vanilla_answer(q)
    print(vanilla)
    print("\n--- RAG- AUGMENTED LLM (retrieved PDF context) ---")
    rag = answer_query(q, index_obj, top_k=3, gpt_model=CHAT_MODEL)
    print(rag)
    comparison_rows.append({
        "Question": q,
        "Vanilla LLM Answer": vanilla,
        "RAG Answer": rag
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


QUESTION 1: What is the main purpose or central argument of this document?

--- VANILLA LLM (no PDF retrieval) ---
I'm sorry, but I cannot access or analyze specific documents. If you can provide some details or key points from the document, I would be happy to help you summarize its main purpose or central argument!

--- RAG- AUGMENTED LLM (retrieved PDF context) ---
The main purpose of the document is to propose a three-year program aimed at integrating NVIDIA's computing and AI platforms into enterprise environments, emphasizing the potential for measurable adoption, operational efficiency, and customer value. It outlines a structured approach to move beyond AI experimentation to deliver impactful solutions, while clarifying that the proposed $200M investment is hypothetical and not an existing public deal.

QUESTION 2: What are the most important methods, steps, or components described?

--- VANILLA LLM (no PDF retrieval) ---
To provide a more accurate response, I would need to kn

,Question,Vanilla LLM Answer,RAG Answer
0,What is the main purpose or central argument o...,"I'm sorry, but I cannot access or analyze spec...",The main purpose of the document is to propose...
1,"What are the most important methods, steps, or...","To provide a more accurate response, I would n...","The most important methods, steps, or componen..."
2,"What key findings, conclusions, or recommendat...","I'm sorry, but I don't have access to specific...",I don't know based on the document.


## 10. Reflection (500–700 words)

After running the three tests, complete the reflection below. Replace every bracketed prompt with observations from your own PDF and actual outputs. Do not claim that RAG improved a result unless your comparison supports that conclusion.

**Reflection draft (edit before submission):**

This assignment involved building a small Retrieval-Augmented Generation (RAG) application that answers questions using information extracted from a PDF. The workflow included PDF parsing, text chunking, embedding generation, vector similarity search, and answer generation through an LLM. I tested the system with three questions and compared a vanilla LLM, which had no access to the uploaded document, with the RAG-augmented version.

For the first question, [briefly describe what the question asked]. The vanilla LLM answered [summarize its answer and note any unsupported or general information]. The RAG system answered [summarize its answer] and used information from the uploaded PDF. For the second question, [describe the question and the key difference between the two answers]. For the third question, [describe the question and comparison]. Across these tests, the main difference I observed was [state your actual finding about accuracy, hallucination, relevance, and completeness]. RAG was especially useful when [give a concrete example from your results]. However, [mention any limitation, such as missing information or an incomplete retrieved passage].

Chunking played an important role because a long PDF cannot efficiently be sent to the language model as one large context. Dividing the text into smaller sections made it possible to retrieve only the passages most related to a question. I used a chunk size of 800 characters and an overlap of 120 characters. The overlap helped preserve information near chunk boundaries, because a sentence or explanation may otherwise be split between two chunks. If chunks are too small, important context can be lost; if they are too large, retrieval may include irrelevant material. A useful next experiment would be to compare different chunk sizes and evaluate which setting retrieves the most complete evidence for the questions.

Embeddings convert text into numerical vectors that represent semantic meaning. This allows the system to find passages related to a question even when the wording is not identical. Cosine similarity ranks the stored vectors by their semantic closeness to the question vector. In this notebook, the highest-ranked chunks are passed to the LLM as context. The quality of the final answer therefore depends on both the embedding model and whether retrieval finds the right passages. If relevant evidence is not retrieved, the LLM may be unable to answer correctly.

The main trade-off was implementation complexity versus answer grounding. A vanilla LLM call is simpler and usually requires fewer steps. RAG adds PDF parsing, embedding requests, vector storage, and retrieval, which can increase setup time, latency, and API usage. On the other hand, it gives the model access to document-specific information that may not exist in its general training knowledge. [Add your observed speed/cost comparison, if measured.]

I would extend this system by adding caching for repeated questions and embeddings. A cache could avoid recomputing identical embeddings or repeating the same LLM request, reducing latency and cost. I could also explore hybrid retrieval by combining semantic vector search with keyword search, which may help with exact names, dates, and technical terms. A future RAG+KAG approach could represent important entities and relationships from the PDF as a knowledge graph, then combine graph-based facts with retrieved text passages. This may improve multi-step questions that require connecting information from different sections. Overall, this project demonstrated how retrieval can connect an LLM to a private document and make answers more relevant to a specific source, while also showing that retrieval quality and careful evaluation remain essential.

**Word count:** Recheck after replacing the bracketed prompts; keep the final reflection between 500 and 700 words.

## 11. Final submission checklist

- [ ] Notebook runs from top to bottom without errors.
- [ ] A readable PDF was uploaded and text was extracted.
- [ ] Chunks were created with overlap.
- [ ] Embeddings were generated through OpenRouter and the vector index was built.
- [ ] Three domain-specific questions were tested.
- [ ] Both vanilla and RAG answers are visible for all three questions.
- [ ] The comparison and reflection are based on actual outputs.
- [ ] The reflection is 500–700 words.
- [ ] No API key appears in notebook outputs, screenshots, or submitted files.
- [ ] Share the completed notebook with the instructor or submit it through the LMS as required.

**Important compatibility note:** The supplied starter notebook's rubric text mentions OpenAI-style function names and defaults, while the assignment instructions require OpenRouter. This notebook keeps the required function names and uses the OpenAI-compatible SDK with OpenRouter's base URL. The local NumPy vector index is an allowed alternative to Pinecone.